# Fase 1: Configuración del Entorno y Análisis Exploratorio de Datos (EDA)
**Integrantes - Grupo 6:** Mateo Castillo y Christian Salinas  
**Asignatura:** Proyecto Integrador  



### Paso 1: Instalar Spark y configurar las variables de entorno
Como Google Colab no viene con Apache Spark instalado de fábrica, lo primero que tenemos que hacer es descargarlo. Necesitamos tres cosas: Java (porque Spark corre sobre la máquina virtual de Java), el motor de Spark en su versión 3.5.1, y una librería de Python llamada `findspark` que nos ayuda a conectar nuestro código con esa instalación.

Ejecuta la siguiente celda para hacer toda la instalación en el servidor de Google.

In [1]:
# Descargamos Java y Spark silenciosamente
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

# Le decimos a la computadora de Google dónde quedaron guardados Java y Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

# Inicializamos findspark para que Python reconozca los comandos distribuidos
import findspark
findspark.init()
print("¡Setup terminado! Ya tenemos Spark en la nube listo para camellar.")

¡Setup terminado! Ya tenemos Spark en la nube listo para camellar.


### Paso 2: Crear nuestra sesión de Spark
En Big Data, para poder procesar información de forma distribuida necesitamos crear lo que se conoce como una `SparkSession`. Esto viene a ser como el cerebro del proyecto: se encarga de gestionar la memoria y coordinar las tareas. Como estamos en Colab, le ponemos `master("local[*]")` para que aproveche todos los núcleos del procesador que Google nos presta gratis.

In [2]:
from pyspark.sql import SparkSession

# Iniciamos la sesión de Spark de nuestro grupo
spark = SparkSession.builder \
    .appName("EDA_Datos_Climaticos_Grupo6") \
    .master("local[*]") \
    .getOrCreate()

print("Sesión de Spark iniciada con éxito.")
print("Versión activa de Spark:", spark.version)

Sesión de Spark iniciada con éxito.
Versión activa de Spark: 3.5.1


### Paso 3: Conectar Google Drive y cargar el Dataset
Para no estar subiendo el archivo CSV a Colab cada vez que se nos cierre la sesión, lo ideal es guardarlo en una carpeta de nuestro Google Drive.

Al correr esta celda, nos va a saltar un mensaje pidiendo permisos para conectarse a Drive. Le damos que sí, verificamos que la ruta de la carpeta sea la correcta (`Proyecto_Clima`) y cargamos el archivo usando el lector de Spark. Ojo, le ponemos `inferSchema=True` para que Spark intente adivinar solito qué columnas son números y cuáles son texto.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Ruta del archivo en tu Drive
ruta_csv = "/content/drive/MyDrive/Proyecto_Clima/GlobalLandTemperaturesByCountry.csv"

# Cargamos los datos en un DataFrame de Spark
df_clima = spark.read.csv(ruta_csv, header=True, inferSchema=True)

print("--- DATOS CARGADOS CON ÉXITO ---")
print(f"Total de registros a procesar: {df_clima.count():,}")
print("\nPrimeras 5 filas del dataset:")
df_clima.show(5)

Mounted at /content/drive
--- DATOS CARGADOS CON ÉXITO ---
Total de registros a procesar: 577,462

Primeras 5 filas del dataset:
+----------+------------------+-----------------------------+-------+
|        dt|AverageTemperature|AverageTemperatureUncertainty|Country|
+----------+------------------+-----------------------------+-------+
|1743-11-01|4.3839999999999995|                        2.294|  Åland|
|1743-12-01|              NULL|                         NULL|  Åland|
|1744-01-01|              NULL|                         NULL|  Åland|
|1744-02-01|              NULL|                         NULL|  Åland|
|1744-03-01|              NULL|                         NULL|  Åland|
+----------+------------------+-----------------------------+-------+
only showing top 5 rows



### Paso 4: Análisis Exploratorio e Inspección del Schema
Ahora que ya cargamos más de medio millón de filas sin que la computadora sufra, toca ver cómo están estructuradas las columnas. Con `printSchema()` vamos a revisar los tipos de datos (si la fecha la tomó como texto, si las temperaturas son decimales, etc.).

Además, como este dataset recopila datos desde el año 1750, es fijote que faltan un montón de mediciones de cuando no había tecnología. Vamos a hacer un conteo automático de valores nulos (`NaN` o `Null`) en cada columna. Esto nos servirá como la justificación perfecta de por qué necesitamos armar un pipeline ETL en la Fase 2 para limpiar toda esa información.

In [4]:
from pyspark.sql.functions import col, sum as _sum

print("1. Estructura de las columnas (Schema):")
df_clima.printSchema()

print("\n2. Radiografía de valores nulos en el dataset:")
# Mapeamos cada columna para contar cuántos registros vacíos tiene
df_clima.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_clima.columns]).show()

print("\n3. Resumen estadístico básico de las temperaturas:")
df_clima.describe(["AverageTemperature"]).show()

1. Estructura de las columnas (Schema):
root
 |-- dt: date (nullable = true)
 |-- AverageTemperature: double (nullable = true)
 |-- AverageTemperatureUncertainty: double (nullable = true)
 |-- Country: string (nullable = true)


2. Radiografía de valores nulos en el dataset:
+---+------------------+-----------------------------+-------+
| dt|AverageTemperature|AverageTemperatureUncertainty|Country|
+---+------------------+-----------------------------+-------+
|  0|             32651|                        31912|      0|
+---+------------------+-----------------------------+-------+


3. Resumen estadístico básico de las temperaturas:
+-------+------------------+
|summary|AverageTemperature|
+-------+------------------+
|  count|            544811|
|   mean| 17.19335423293583|
| stddev|10.953966445121187|
|    min|           -37.658|
|    max| 38.84200000000001|
+-------+------------------+



### Conclusiones del EDA de esta Fase I
Con los resultados que obtuvimos en pantalla, podemos ver que:
1. El dataset tiene el volumen ideal para Big Data (más de 570k filas), pero se deja manejar súper rápido con Spark.
2. Hay una cantidad importante de nulos en `AverageTemperature` (la columna que queremos predecir). Esto pasa porque en los siglos XVIII y XIX las mediciones climáticas eran escasas.
3. **Próximo paso obligatorio:** Para la Fase II nos va a tocar limpiar estos nulos (ya sea borrando los años viejos o imputando los promedios con PySpark) y pasar el archivo final a formato **Parquet** para dejar la mesa servida antes de entrenar el modelo de Machine Learning con Spark MLlib.